# Homework 05 — VaR of a Market-Cap-Weighted Portfolio (MSFT, AAPL, GOOGL)

The derivations of the loss operators (Q3) and of the analytic linear VaR (Q5), and the weight calculation (Q2), are handwritten. This notebook contains the data, the numerical results, and the written answers for Q1, Q3–Q6. All numbers in the text cells are generated from the computed results.

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from scipy.stats import norm
from IPython.display import Markdown, display

pd.set_option("display.float_format", "{:,.4f}".format)

tickers = ["MSFT", "AAPL", "GOOGL"]
V0      = 1_000_000      # portfolio value ($)
alpha   = 0.95
N_sim   = 100_000
lam = theta = 0.97
rng = np.random.default_rng(2026)

def sci(x, d=4):
    # LaTeX scientific notation, e.g. -1.0889 \times 10^{-3}
    m, e = f"{x:.{d}e}".split("e")
    return rf"{m} \times 10^{{{int(e)}}}"

def money(x, d=0):
    # dollar amount with the sign in front of the dollar sign
    return ("-" if x < 0 else "") + f"\\${abs(x):,.{d}f}"


## Q1. Daily closing prices, 05/23/2024 – 05/22/2026

Daily closing prices (the `Close` field, not adjusted for dividends) are downloaded from Yahoo Finance with the `yfinance` package. Only dates on which all three stocks have a closing price are kept.

In [ ]:
# yfinance treats `end` as exclusive, so end = 05/23/2026 includes 05/22/2026
raw = yf.download(tickers, start="2024-05-23", end="2026-05-23",
                  auto_adjust=False, progress=False)
S = raw["Close"][tickers].dropna()
S.to_csv("prices.csv")          # local copy for reproducibility

print("First five observations:")
display(S.head())
print("Last five observations:")
display(S.tail())
print(f"Total number of observations: {len(S)}")

In [ ]:
display(Markdown(
    f"**Answer.** The dataset covers {S.index[0]:%m/%d/%Y} to {S.index[-1]:%m/%d/%Y} and contains "
    f"**{len(S)} daily observations** for each stock, which gives {len(S) - 1} daily log returns. "
    f"The first five and last five observations are shown above."))

## Q2. Market-capitalization weights $\omega_i = \lambda^{(i)}_t S^{(i)}_t / V_t$

In [ ]:
mcap = pd.Series({"MSFT": 3.109, "AAPL": 4.536, "GOOGL": 4.64})  # $ trillion, as of 05/22/2026
w = mcap / mcap.sum()

# Number of shares lambda_i = w_i * V / S_i, using the 05/22/2026 close
S_t = S.iloc[-1]
shares = w * V0 / S_t

print(f"Total market cap: ${mcap.sum():.3f} trillion")
display(pd.DataFrame({"Market cap ($T)": mcap, "Weight w_i": w,
                      "Price S_t": S_t, "Shares lambda_i": shares}))
w = w.values

## Q3. Empirical VaR under $l_{[t]}$, $l^{\mathrm{lin}}_{[t]}$ and $l^{\mathrm{quad}}_{[t]}$

The risk factors are the log prices, so the risk-factor changes are the daily log returns $X_{t+1} = \ln S_{t+1} - \ln S_t$. With $V_t = \$1{,}000{,}000$ and weights $\omega$ from Q2, the loss operators (derived by hand) are

$$l_{[t]}(x) = -V_t \sum_{i=1}^3 \omega_i \left(e^{x_i} - 1\right), \qquad
l^{\mathrm{lin}}_{[t]}(x) = -V_t \sum_{i=1}^3 \omega_i x_i, \qquad
l^{\mathrm{quad}}_{[t]}(x) = -V_t \sum_{i=1}^3 \omega_i \left(x_i + \tfrac{1}{2}x_i^2\right).$$

Each historical return vector $x_1, \dots, x_n$ is treated as a possible scenario for tomorrow. The empirical VaR is the empirical $\alpha$-quantile of the scenario losses: sorting $L_{(1)} \le \dots \le L_{(n)}$, $\widehat{\mathrm{VaR}}_{0.95} = L_{(\lceil 0.95\,n \rceil)}$.

In [ ]:
X = np.log(S).diff().dropna()          # daily log returns
Xv = X.values
print(f"Number of log-return observations: {len(X)}")

def loss_full(x):  return -V0 * ((np.exp(x) - 1) @ w)
def loss_lin(x):   return -V0 * (x @ w)
def loss_quad(x):  return -V0 * ((x + 0.5 * x**2) @ w)

def emp_var(L, a=alpha):
    # Empirical VaR: the ceil(n * a)-th smallest loss
    Ls = np.sort(L)
    return Ls[int(np.ceil(len(Ls) * a)) - 1]

ops = ["Full", "Linear", "Quadratic"]
L_hist_full = loss_full(Xv)
hist_var = pd.Series([emp_var(L_hist_full), emp_var(loss_lin(Xv)), emp_var(loss_quad(Xv))],
                     index=ops, name="Historical VaR 95% ($)")
hist_var.to_frame()

In [ ]:
k = int(np.ceil(len(Xv) * alpha))
display(Markdown(
    f"**Answer.** With n = {len(Xv)} historical scenarios, the VaR is the {k}th smallest loss. "
    f"The 95% one-day VaR of the \\$1,000,000 portfolio is "
    f"**\\${hist_var['Full']:,.2f}** under $l_{{[t]}}$, "
    f"**\\${hist_var['Linear']:,.2f}** under $l^{{\\mathrm{{lin}}}}_{{[t]}}$, and "
    f"**\\${hist_var['Quadratic']:,.2f}** under $l^{{\\mathrm{{quad}}}}_{{[t]}}$."))

## Q4. EWMA estimates of $\mu$ and $\Sigma$

Starting from $\mu_0 = 0$ and $\Sigma_0 = 0$, the recursions

$$\mu_{t+\Delta} = \lambda \mu_t + (1-\lambda) X_t, \qquad
\Sigma_{t+\Delta} = \theta \Sigma_t + (1-\theta)(X_t - \mu_t)(X_t - \mu_t)^{\top}, \qquad \lambda = \theta = 0.97,$$

are run over all daily log returns in chronological order. The final values are the estimates as of 05/22/2026 and are used for the next period, 05/22/2026 → 05/25/2026. An observation $k$ days old receives weight $(1-\lambda)\lambda^{k}$, so recent days dominate; the weight left on the zero starting values is $\lambda^{n}$, which is negligible here.

In [ ]:
mu    = np.zeros(3)          # mu_0 = 0
Sigma = np.zeros((3, 3))     # Sigma_0 = 0

for x in Xv:                 # chronological; the last x is the 05/22/2026 return
    d = x - mu               # uses mu_t before the update
    mu    = lam   * mu    + (1 - lam)   * x
    Sigma = theta * Sigma + (1 - theta) * np.outer(d, d)

print("EWMA mean vector mu (used for 05/22/2026 -> 05/25/2026):")
display(pd.Series(mu, index=tickers).to_frame("mu"))
print("EWMA covariance matrix Sigma:")
display(pd.DataFrame(Sigma, index=tickers, columns=tickers).style.format("{:.4e}"))
print("Implied daily volatilities:", np.sqrt(np.diag(Sigma)).round(5))

In [ ]:
def latex_vec(v):
    return r"\begin{pmatrix}" + r"\\ ".join(sci(x) for x in v) + r"\end{pmatrix}"
def latex_mat(M):
    rows = [" & ".join(sci(x) for x in r) for r in M]
    return r"\begin{pmatrix}" + r"\\ ".join(rows) + r"\end{pmatrix}"

vols = np.sqrt(np.diag(Sigma))
corr = Sigma / np.outer(vols, vols)
display(Markdown(
    "**Answer.** With the order (MSFT, AAPL, GOOGL), the EWMA estimates as of 05/22/2026 are\n\n"
    f"$$\\hat\\mu = {latex_vec(mu)}, \\qquad \\hat\\Sigma = {latex_mat(Sigma)}.$$\n\n"
    f"The implied daily volatilities are {vols[0]:.4f} (MSFT), {vols[1]:.4f} (AAPL) and {vols[2]:.4f} (GOOGL), "
    f"i.e. about {vols[0]*np.sqrt(252):.1%}, {vols[1]*np.sqrt(252):.1%} and {vols[2]*np.sqrt(252):.1%} annualized. "
    f"The implied correlations are {corr[0,1]:.3f} (MSFT–AAPL), {corr[0,2]:.3f} (MSFT–GOOGL) and {corr[1,2]:.3f} (AAPL–GOOGL). "
    f"The weight remaining on the zero initial values is $0.97^{{{len(Xv)}}} = {sci(lam**len(Xv), 1)}$, so the initialization has no practical effect."))

## Q5. VaR under $X \sim N(\mu, \Sigma)$: Monte Carlo for $l_{[t]}$ and $l^{\mathrm{quad}}_{[t]}$, analytic for $l^{\mathrm{lin}}_{[t]}$

**Linear operator (analytic).** Since $l^{\mathrm{lin}}_{[t]}(X) = -V_t\,\omega^{\top}X$ is a linear function of a normal vector, it is normal with mean $-V_t\,\omega^{\top}\mu$ and variance $V_t^2\,\omega^{\top}\Sigma\,\omega$ (derivation handwritten), so

$$\mathrm{VaR}^{\mathrm{lin}}_{\alpha} = -V_t\,\omega^{\top}\mu + V_t\sqrt{\omega^{\top}\Sigma\,\omega}\;\Phi^{-1}(\alpha).$$

**Full and quadratic operators (Monte Carlo).** These losses are nonlinear in $X$ and have no closed-form distribution. We draw $N = 100{,}000$ vectors $X^{(1)}, \dots, X^{(N)} \sim N(\hat\mu, \hat\Sigma)$, evaluate the loss operator at each draw, and take the empirical 95% quantile, i.e. the $95{,}000$-th smallest simulated loss.

In [ ]:
Z = rng.multivariate_normal(mu, Sigma, size=N_sim)

L_norm_full = loss_full(Z)
var_full_mc = emp_var(L_norm_full)
var_quad_mc = emp_var(loss_quad(Z))

# Linear operator, analytic: VaR = -V w'mu + V sqrt(w' Sigma w) * Phi^{-1}(alpha)
sd_p = np.sqrt(w @ Sigma @ w)
var_lin_an = -V0 * (w @ mu) + V0 * sd_p * norm.ppf(alpha)

print(f"w'mu = {w @ mu:.6e},  sqrt(w' Sigma w) = {sd_p:.6f},  Phi^-1(0.95) = {norm.ppf(alpha):.4f}")
normal_var = pd.Series([var_full_mc, var_lin_an, var_quad_mc],
                       index=["Full (MC)", "Linear (analytic)", "Quadratic (MC)"],
                       name="Normal-EWMA VaR 95% ($)")
normal_var.to_frame()

In [ ]:
var_lin_mc = emp_var(loss_lin(Z))      # sanity check of the analytic formula
display(Markdown(
    "**Answer.** Substituting the EWMA estimates into the analytic formula:\n\n"
    f"$$\\mathrm{{VaR}}^{{\\mathrm{{lin}}}}_{{0.95}} = -10^6 \\times ({sci(w @ mu)}) + 10^6 \\times {sd_p:.6f} \\times {norm.ppf(alpha):.4f} "
    f"= {-V0 * (w @ mu):,.2f} + {V0 * sd_p * norm.ppf(alpha):,.2f} = {var_lin_an:,.2f}.$$\n\n"
    f"The Monte Carlo estimates are **\\${var_full_mc:,.2f}** under $l_{{[t]}}$ and **\\${var_quad_mc:,.2f}** under $l^{{\\mathrm{{quad}}}}_{{[t]}}$. "
    f"As a check, applying the same simulation to $l^{{\\mathrm{{lin}}}}_{{[t]}}$ gives \\${var_lin_mc:,.2f}, "
    f"within {abs(var_lin_mc / var_lin_an - 1):.2%} of the analytic value, so the simulation size is adequate."))

## Q6. Comparison

In [ ]:
summary = pd.DataFrame({
    "Historical":    hist_var.values,
    "Normal (EWMA)": normal_var.values,
}, index=ops)
summary["Normal / Historical"] = summary["Normal (EWMA)"] / summary["Historical"]

r_p = Xv @ w   # linearized portfolio log return, for volatility comparison
print(f"Portfolio daily vol:  full sample = {r_p.std(ddof=1):.5f},  EWMA = {sd_p:.5f}")
print(f"Historical portfolio returns: skewness = {pd.Series(r_p).skew():.3f}, "
      f"excess kurtosis = {pd.Series(r_p).kurt():.3f}")
summary

In [ ]:
BLUE, ORANGE, INK, MUTED = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e"

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# (a) VaR estimates side by side
xpos, bw = np.arange(len(ops)), 0.36
b1 = ax1.bar(xpos - bw/2 - 0.01, summary["Historical"],    bw, color=BLUE,   label="Historical")
b2 = ax1.bar(xpos + bw/2 + 0.01, summary["Normal (EWMA)"], bw, color=ORANGE, label="Normal (EWMA)")
for bars in (b1, b2):
    ax1.bar_label(bars, fmt="${:,.0f}", fontsize=8, color=INK, padding=2)
ax1.set_xticks(xpos, ["Full", "Linear", "Quadratic"])
ax1.set_ylabel("95% one-day VaR ($)")
ax1.set_title("(a) VaR by method and loss operator")
ax1.set_ylim(0, summary.values[:, :2].max() * 1.25)
ax1.legend(frameon=False, ncol=2, loc="upper left")
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:,.0f}"))

# (b) Loss distributions under the full operator
bins = np.linspace(min(L_hist_full.min(), np.percentile(L_norm_full, 0.01)),
                   max(L_hist_full.max(), np.percentile(L_norm_full, 99.99)), 60)
ax2.hist(L_hist_full, bins=bins, density=True, color=BLUE, alpha=0.55, label="Historical losses")
ax2.hist(L_norm_full, bins=bins, density=True, histtype="step", lw=2, color=ORANGE,
         label="Normal-EWMA simulated losses")
ax2.axvline(summary.loc["Full", "Historical"],    color=BLUE,   ls="--", lw=1.5, label="Historical VaR")
ax2.axvline(summary.loc["Full", "Normal (EWMA)"], color=ORANGE, ls="--", lw=1.5, label="Normal-EWMA VaR")
ax2.set_xlabel("One-day loss ($)")
ax2.set_ylabel("Density")
ax2.set_yticks([])
ax2.set_title("(b) Loss distributions, full operator")
ax2.legend(frameon=False, fontsize=8, loc="upper left")

for ax in (ax1, ax2):
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(colors=MUTED)
    ax.grid(axis="y", color="#e6e5e1", lw=0.8)
    ax.set_axisbelow(True)
ax2.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:,.0f}"))

plt.tight_layout()
plt.savefig("hw05_var_comparison.png", dpi=200)
plt.show()

### Discussion

In [ ]:
H, Nn = summary["Historical"], summary["Normal (EWMA)"]
vol_h, vol_e = r_p.std(ddof=1), sd_p
kurt, skew = pd.Series(r_p).kurt(), pd.Series(r_p).skew()
z = norm.ppf(alpha)

# Decomposition of the gap: normal model with full-sample parameters
var_norm_hist = V0 * (-r_p.mean() + vol_h * z)

ratio = Nn["Full"] / H["Full"] - 1
material = abs(ratio) > 0.10
spread_h = (H.max() - H.min()) / H["Full"]
spread_n = (Nn.max() - Nn.min()) / Nn["Full"]

p1 = (f"**Loss operators.** Within each method the three operators give almost the same VaR: they differ by at most "
      f"{spread_h:.2%} (historical) and {spread_n:.2%} (normal-EWMA). The linear VaR is the largest because "
      f"$e^x \\ge 1 + x$ implies $l^{{\\mathrm{{lin}}}}_{{[t]}}(x) \\ge l_{{[t]}}(x)$ for every scenario; the gap is "
      f"$V_t\\sum_i \\omega_i (e^{{x_i}} - 1 - x_i) \\approx V_t\\sum_i \\omega_i x_i^2/2$, which is only a few hundred dollars "
      f"for daily moves of 1–3%. The quadratic operator matches the full operator to within "
      f"\\${abs(H['Quadratic'] - H['Full']):,.2f} (historical) and \\${abs(Nn['Quadratic'] - Nn['Full']):,.2f} (normal), "
      f"because its error is of order $x^3$. For a portfolio of stocks over one day, the choice of loss operator is immaterial.")

if material:
    p2 = (f"**Historical vs. normal-EWMA.** The two methods differ materially: the normal-EWMA VaR is "
          f"{abs(ratio):.1%} {'higher' if ratio > 0 else 'lower'} than the historical VaR under the full operator "
          f"(\\${Nn['Full']:,.0f} vs. \\${H['Full']:,.0f}).")
else:
    p2 = (f"**Historical vs. normal-EWMA.** The two methods do not differ materially: the normal-EWMA VaR is only "
          f"{abs(ratio):.1%} {'higher' if ratio > 0 else 'lower'} than the historical VaR under the full operator "
          f"(\\${Nn['Full']:,.0f} vs. \\${H['Full']:,.0f}).")

vol_eff   = V0 * z * (vol_e - vol_h)          # EWMA vs full-sample volatility
mean_eff  = -V0 * (w @ mu - r_p.mean())       # EWMA vs full-sample mean
shape_eff = var_norm_hist - H["Linear"]       # normal vs empirical shape, same data

p3 = (f"To locate the gap, use a normal model with the full-sample mean and volatility as a bridge; its linear VaR is "
      f"{money(var_norm_hist)}. Moving from the historical VaR ({money(H['Linear'])}) to the normal-EWMA VaR "
      f"({money(var_lin_an)}) then splits into three parts:\n\n"
      f"- *Volatility* ({money(vol_eff)}): the EWMA portfolio volatility is {vol_e:.4f}, versus {vol_h:.4f} over the "
      f"two-year sample. EWMA with $\\theta = 0.97$ has an effective memory of about $1/(1-\\theta) \\approx 33$ trading days, "
      f"so it reflects the {'more volatile' if vol_e > vol_h else 'calmer'} recent period rather than the two-year average.\n"
      f"- *Mean* ({money(mean_eff)}): the EWMA mean is also dominated by the last few weeks. Its contribution "
      f"$-V_t\\,\\omega^\\top\\hat\\mu$ is {money(-V0 * (w @ mu))}, compared with {money(-V0 * r_p.mean())} for the full-sample mean. "
      f"A daily mean estimated from a short window is very noisy, so this part should not be over-interpreted.\n"
      f"- *Distribution shape* ({money(shape_eff)}): the historical portfolio returns have skewness {skew:.2f} and excess "
      f"kurtosis {kurt:.2f}. "
      + ("Positive excess kurtosis means a more peaked centre and fatter tails than the normal distribution; "
         if kurt > 0 else "The excess kurtosis is not positive, so the empirical tails are not heavier than normal; ")
      + f"this part measures how much the normal shape alone changes the 95% quantile. The normal assumption would understate "
      f"risk more seriously at higher levels such as 99%.\n\n"
      f"In addition, the historical VaR rests on only about {int(round((1 - alpha) * len(Xv)))} tail observations, so it has "
      f"sizeable estimation error of its own.")

parts = {"the volatility estimate (recent EWMA versus two-year average)": vol_eff,
         "the mean estimate": mean_eff,
         "the normality assumption versus the empirical distribution": shape_eff}
order = sorted(parts, key=lambda k: abs(parts[k]), reverse=True)
p4 = (f"**Conclusion.** The loss operator hardly matters; the choice of method "
      f"{'does' if material else 'matters more, though here not materially'}. "
      f"The largest contribution to the difference comes from {order[0]} ({money(parts[order[0]])}), "
      f"followed by {order[1]} ({money(parts[order[1]])}) and {order[2]} ({money(parts[order[2]])}).")

display(Markdown("\n\n".join([p1, p2, p3, p4])))